# ChipChat-Style Adder Generation and Verification Notebook
This notebook follows the professor's ChipChat workflow for two adders: RCA8 and CLA8.

In [11]:
# STEP 0: Insert your API key if using OpenAI/ChipChat-like generation
OPENAI_API_KEY = " "


## Step 1: Selected Adders
- RCA8: simple ripple-carry baseline.
- CLA8: faster carry-lookahead architecture using generate/propagate logic.

In [12]:
from pathlib import Path
print("RCA8 golden preview:")
print(Path("golden/RCA8.v").read_text()[:500])
print("\nCLA8 golden preview:")
print(Path("golden/CLA8.v").read_text()[:500])


RCA8 golden preview:

// Full Adder
module FA(output sum, cout, input a, b, cin);
  wire w0, w1, w2;
  
  xor  (w0, a, b);
  xor  (sum, w0, cin);
  
  and  (w1, w0, cin);
  and  (w2, a, b);
  or  (cout, w1, w2);
endmodule

// Ripple Carry Adder - 8 bits
module RCA8(output [7:0] sum, output cout, input [7:0] a, b);
  
  wire [7:1] c;
  
  FA fa0(sum[0], c[1], a[0], b[0], 0);
  FA fa[6:1](sum[6:1], c[7:2], a[6:1], b[6:1], c[6:1]);
  FA fa7(sum[7], cout, a[7], b[7], c[7]);
  
endmodule

CLA8 golden preview:

module PGGen(output g, p, input a, b);
 
  and (g, a, b);
  xor (p, a, b);
 
endmodule

module CLA8(output [7:0] sum, output cout, input [7:0] a, b);
wire [7:0] g, p, c;
wire [35:0] e;
wire cin;

buf (cin, 0);
PGGen pggen[7:0](g[7:0],p[7:0],a[7:0],b[7:0]);

//c[0]
and (e[0], cin, p[0]);
or (c[0], e[0], g[0]);

//c[1]
and (e[1], cin, p[0], p[1]);
and (e[2], g[0], p[1]);
or (c[1], e[1], e[2], g[1]);

//c[2]
and (e[3], cin, p[0], p[1], p[2]);
and (e[4], g[0], p[1], p[2]);
and (e[5], g[1]

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Prompt Used for Natural-Language Description

In [15]:
description_prompt = """Analyze the following Verilog code and provide a detailed natural language description of the design. Include: (1) Overall architecture and purpose, (2) Module hierarchy and interfaces, (3) Signal flow and datapath, (4) Key logic operations, (5) Any special design features. Be specific about how the circuit implements its functionality."""
print(description_prompt)


Analyze the following Verilog code and provide a detailed natural language description of the design. Include: (1) Overall architecture and purpose, (2) Module hierarchy and interfaces, (3) Signal flow and datapath, (4) Key logic operations, (5) Any special design features. Be specific about how the circuit implements its functionality.


## Step 3: Prompt Used for Verilog Regeneration

In [16]:
generation_prompt = """Based on the following description, generate Verilog code that implements this exact architecture. Maintain the same module hierarchy, signal names, and design approach described. Use structural Verilog with gate-level primitives where specified. Return only Verilog code."""
print(generation_prompt)


Based on the following description, generate Verilog code that implements this exact architecture. Maintain the same module hierarchy, signal names, and design approach described. Use structural Verilog with gate-level primitives where specified. Return only Verilog code.


## Step 4: Simulation Commands

In [24]:
!apt-get update -qq
!apt-get install -y iverilog yosys

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common librsvg2-common
  libxcomposite1 libxtst6 python3-cairo python3-gi-cairo python3-numpy
  session-migration xdot
Suggested packages:
  gtkwave gvfs python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas iverilog libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common
  librsvg2-common libxcomposite1 libxts

In [25]:
!mkdir -p logs
!iverilog -o logs/rca8_sim generated/RCA8_llm_generated.v testbenches/RCA8_tb.v
!cd logs && vvp rca8_sim > RCA8_simulation_output.txt
!tail -5 logs/RCA8_simulation_output.txt

!iverilog -o logs/cla8_sim generated/CLA8_llm_generated.v testbenches/CLA8_tb.v
!cd logs && vvp cla8_sim > CLA8_simulation_output.txt
!tail -5 logs/CLA8_simulation_output.txt


PASS RCA8: a=cc b=33 sum=ff cout=0 c=0000000
PASS RCA8: a=dd b=22 sum=ff cout=0 c=0000000
PASS RCA8: a=ee b=11 sum=ff cout=0 c=0000000
PASS RCA8: a=ff b=00 sum=ff cout=0 c=0000000
SUMMARY RCA8: ALL TESTS PASSED
PASS CLA8: a=cc b=33 sum=ff cout=0 g=00000000 p=11111111 c=00000000
PASS CLA8: a=dd b=22 sum=ff cout=0 g=00000000 p=11111111 c=00000000
PASS CLA8: a=ee b=11 sum=ff cout=0 g=00000000 p=11111111 c=00000000
PASS CLA8: a=ff b=00 sum=ff cout=0 g=00000000 p=11111111 c=00000000
SUMMARY CLA8: ALL TESTS PASSED


## Step 5: Yosys Synthesis Commands

In [28]:
%%writefile scripts/run_yosys.py
import subprocess, re, sys, json, os

def synthesize(verilog_file, top_module):
    verilog_file = os.path.abspath(verilog_file)

    ys_script = f"""
read_verilog {verilog_file}
hierarchy -check -top {top_module}
proc
opt
techmap
opt
abc
opt
stat
"""

    with open("scripts/temp_synth.ys", "w") as f:
        f.write(ys_script)

    proc = subprocess.run(
        ["yosys", "-s", "scripts/temp_synth.ys"],
        capture_output=True,
        text=True
    )

    log = proc.stdout + proc.stderr

    if proc.returncode != 0:
        raise RuntimeError(log)

    cells = re.search(r"Number of cells:\s+(\d+)", log)
    ppa = {
        "cell_count": int(cells.group(1)) if cells else None,
        "logic_levels": "N/A with generic abc flow",
        "area_um2": "N/A without liberty file"
    }

    return ppa, log

if __name__ == "__main__":
    ppa, log = synthesize(sys.argv[1], sys.argv[2])
    print(json.dumps(ppa, indent=2))

Overwriting scripts/run_yosys.py


In [30]:
!python scripts/run_yosys.py generated/RCA8_llm_generated.v RCA8
!python scripts/run_yosys.py generated/CLA8_llm_generated.v CLA8


{
  "cell_count": 5,
  "logic_levels": "N/A with generic abc flow",
  "area_um2": "N/A without liberty file"
}
{
  "cell_count": 29,
  "logic_levels": "N/A with generic abc flow",
  "area_um2": "N/A without liberty file"
}


## Step 6: Yosys Equivalence Check

In [31]:
!cd scripts && yosys -s equiv_check_rca8.ys
!cd scripts && yosys -s equiv_check_cla8.ys



 /----------------------------------------------------------------------------\
 |                                                                            |
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |                                                                            |
 |  Copyright (C) 2012 - 2019  Clifford Wolf <clifford@clifford.at>           |
 |                                                                            |
 |  Permission to use, copy, modify, and/or distribute this software for any  |
 |  purpose with or without fee is hereby granted, provided that the above    |
 |  copyright notice and this permission notice appear in all copies.         |
 |                                                                            |
 |  THE SOFTWARE IS PROVIDED "AS IS" AND THE AUTHOR DISCLAIMS ALL WARRANTIES  |
 |  WITH REGARD TO THIS SOFTWARE INCLUDING ALL IMPLIED WARRANTIES OF          |
 |  MERCHANTABILITY AND FITNESS. IN NO 